# Recipients by sector

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

In [ ]:
recipients_sector_path = Path("../../../resources/data/restricted/defensoria/destinatario_por_sector.csv")

df = pd.read_csv(recipients_sector_path, sep=",", encoding="utf-8-sig")
df.columns = df.columns.str.strip()

text_columns = ["Destinatario", "Destinatario por sector"]
df[text_columns] = df[text_columns].apply(lambda column: column.str.strip())

display(df.head(10))
print(f"Rows: {len(df):,}")
print(f"Unique sectors: {df['Destinatario por sector'].nunique():,}")
print(f"Unique recipients: {df['Destinatario'].nunique():,}")

## Most frequent sectors

Number of rows for each `Destinatario por sector` value, sorted from highest to lowest.

In [ ]:
sector_frequency = (
    df.groupby("Destinatario por sector", dropna=False)
    .size()
    .reset_index(name="cantidad_destinatarios")
    .sort_values("cantidad_destinatarios", ascending=False)
    .reset_index(drop=True)
)

sector_frequency["porcentaje"] = (
    sector_frequency["cantidad_destinatarios"].div(len(df)).mul(100).round(1)
)

display(sector_frequency)

In [ ]:
ax = (
    sector_frequency
    .sort_values("cantidad_destinatarios")
    .plot.barh(
        x="Destinatario por sector",
        y="cantidad_destinatarios",
        figsize=(10, 7),
        legend=False,
        title="Most frequent sectors",
    )
)
ax.set_xlabel("Number of recipients")
ax.set_ylabel("Sector")

## Recipients grouped by sector

Frequency of each `Destinatario` within each `Destinatario por sector` value.

In [ ]:
destinatario_by_sector = (
    df.groupby(["Destinatario por sector", "Destinatario"], dropna=False)
    .size()
    .reset_index(name="cantidad")
    .sort_values(
        ["Destinatario por sector", "cantidad", "Destinatario"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

display(destinatario_by_sector)

## Top recipients within each sector

The five most repeated recipients per sector, useful for quickly reviewing which names explain each group's frequency.

In [ ]:
top_destinatarios_by_sector = destinatario_by_sector.groupby(
    "Destinatario por sector",
    group_keys=False,
).head(5)

display(top_destinatarios_by_sector)

## Cargo lookup from GCBA organigram

Flatten the organigram tree so each person with a `nombre` keeps their own `cargo` and the parent `cargo` from the dictionary that contains them in `dependencias`.

In [ ]:
import json
import unicodedata


organigram_path = Path("organigram_GCBA.json")

with organigram_path.open(encoding="utf-8") as f:
    organigram = json.load(f)


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value)
    return "".join(char for char in value if not unicodedata.combining(char))


def iter_people_with_dependency(node, parent=None, path=None):
    if path is None:
        path = []

    if isinstance(node, list):
        for child in node:
            yield from iter_people_with_dependency(child, parent=parent, path=path)
        return

    if not isinstance(node, dict):
        return

    cargo = str(node.get("cargo", "")).strip()
    sigla = str(node.get("sigla", "")).strip()
    nombre = str(node.get("nombre", "")).strip()
    current_path = [*path, cargo] if cargo else path

    if nombre:
        yield {
            "nombre": nombre,
            "cargo": cargo,
            "sigla": sigla,
            "depende_de_cargo": (parent or {}).get("cargo"),
            "depende_de_sigla": (parent or {}).get("sigla"),
            "ruta_cargos": " > ".join(current_path),
        }

    current = {"cargo": cargo, "sigla": sigla}
    for child in node.get("dependencias", []):
        yield from iter_people_with_dependency(child, parent=current, path=current_path)


organigram_people = pd.DataFrame(iter_people_with_dependency(organigram))

display(organigram_people.head(20))
print(f"People with nombre: {len(organigram_people):,}")

In [ ]:
def search_organigram_people(query, field="nombre"):
    query_normalized = normalize_text(query)
    field_normalized = organigram_people[field].map(normalize_text)
    return organigram_people[field_normalized.str.contains(query_normalized, na=False)].reset_index(drop=True)


# Search by person name.
display(search_organigram_people("Leonardo Coppola"))

# Search by cargo, useful when checking a sector such as Vivienda.
display(search_organigram_people("Cultura", field="cargo"))

## Direct JSON lookup

In [ ]:
def find_people_in_organigram(node, query, field="nombre", parent=None, path=None):
    if path is None:
        path = []

    if isinstance(node, list):
        for child in node:
            yield from find_people_in_organigram(child, query, field=field, parent=parent, path=path)
        return

    if not isinstance(node, dict):
        return

    cargo = str(node.get("cargo", "")).strip()
    sigla = str(node.get("sigla", "")).strip()
    nombre = str(node.get("nombre", "")).strip()
    current_path = [*path, cargo] if cargo else path

    searchable_value = {
        "nombre": nombre,
        "cargo": cargo,
        "sigla": sigla,
    }.get(field, "")

    if normalize_text(query) in normalize_text(searchable_value):
        yield {
            "nombre": nombre,
            "cargo": cargo,
            "sigla": sigla,
            "depende_de_cargo": (parent or {}).get("cargo"),
            "depende_de_sigla": (parent or {}).get("sigla"),
            "ruta_cargos": " > ".join(current_path),
        }

    current = {"cargo": cargo, "sigla": sigla}
    for child in node.get("dependencias", []):
        yield from find_people_in_organigram(child, query, field=field, parent=current, path=current_path)


list(find_people_in_organigram(organigram, "Leonardo Coppola", field="nombre"))

In [ ]:
list(find_people_in_organigram(organigram, "Tobias", field="nombre"))

## Fuzzy search

`search_organigram_people` and `find_people_in_organigram` only match literal substrings, so a typo or a slightly different wording (e.g. missing accents, abbreviations, extra/missing words) returns nothing. `search_organigram_people_fuzzy` scores every row against the query with `rapidfuzz` and keeps the closest matches, ranked by similarity.

In [ ]:
from rapidfuzz import fuzz, process


def search_organigram_people_fuzzy(query, field="nombre", limit=10, score_cutoff=70):
    query_normalized = normalize_text(query)
    choices = organigram_people[field].map(normalize_text)

    matches = process.extract(
        query_normalized,
        choices,
        scorer=fuzz.WRatio,
        limit=limit,
        score_cutoff=score_cutoff,
    )

    matched_indices = [index for _, _, index in matches]
    scores = {index: score for _, score, index in matches}

    result = organigram_people.loc[matched_indices].copy()
    result["match_score"] = result.index.map(scores)
    return result.sort_values("match_score", ascending=False).reset_index(drop=True)


display(search_organigram_people_fuzzy("Leonrdo Copola"))

display(search_organigram_people_fuzzy("Cultur", field="cargo"))